# Code Generator

The requirement: use a Frontier model to generate high performance Golang code from Python code


In [1]:
import os
import io
import sys
import subprocess
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-


In [3]:
# Connect to client libraries

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)

models = [
    "gpt-4o",
    "claude-sonnet-4-5-20250929",
    "grok-4",
    "gemini-2.5-pro",
    "qwen2.5-coder",
    "deepseek-coder-v2",
    "qwen/qwen3-coder-30b-a3b-instruct",
    "openai/gpt-oss-120b",
]

clients = {
    "gpt-4o": openai,
    "claude-sonnet-4-5-20250929": anthropic,
    "grok-4": grok,
    "gemini-2.5-pro": gemini,
    "openai/gpt-oss-120b": groq,
    "qwen2.5-coder": ollama,
    "deepseek-coder-v2": ollama,
    "qwen/qwen3-coder-30b-a3b-instruct": openrouter,
}

In [4]:
# Go compile / run commands 

GO_SOURCE = "main.go"
GO_BINARY = "./main_go"

compile_command = ["go", "build", "-o", GO_BINARY, GO_SOURCE]
run_command = [GO_BINARY]

In [12]:
# Prompts

system_prompt = """
Your task is to convert Python code into high-performance Go (Golang) code.
Respond only with Go code. Do not provide any explanation other than occasional comments.
The Go response needs to produce identical output in the fastest possible time.
Use goroutines and channels where beneficial for parallelism.
Always wrap your logic in a proper main() function with the necessary imports.

IMPORTANT type rules:
- Always use uint64 (never uint32) for any LCG or modular arithmetic involving 2^32 or larger constants.
- For any constant like 2**32, use the Go literal 1<<32 stored as uint64.
- Use int64 for accumulators that may hold large sums (e.g. max subarray totals).
"""


def user_prompt_for(python):
    return f"""
Port this Python code to Go with the fastest possible implementation that produces identical output.
Respond only with Go code — no markdown fences, no explanation.

Python code to port:

```python
{python}
```
"""

def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)},
    ]

In [10]:
# Core functions

def write_output(code):
    with open(GO_SOURCE, "w") as f:
        f.write(code)

def port(model, python):
    client = clients[model]
    response = client.chat.completions.create(model=model, messages=messages_for(python))
    reply = response.choices[0].message.content
    reply = reply.replace("```go", "").replace("```golang", "").replace("```", "").strip()
    return reply

def run_python(code):
    globals_dict = {"__builtins__": __builtins__}
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer
    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout
    return output

def compile_and_run(code):
    write_output(code)
    try:
        compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [7]:
# Sample Python workload

python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value

def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

n = 10000
initial_seed = 42
min_val = -10
max_val = 10

import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [13]:
# Gradio UI

CSS = """
.run-btn { min-width: 120px !important; }
.py  { background: #1e3a5f !important; }
.cpp { background: #00adb5 !important; }
.convert-btn { background: #f0a500 !important; color: #000 !important; font-weight: bold !important; }
.py-out textarea { background: #0d1b2a !important; color: #90caf9 !important; }
.cpp-out textarea { background: #0d2626 !important; color: #80cbc4 !important; }
"""

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title="Port from Python to Go") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (original)",
                value=python_hard,
                language="python",
                lines=26,
            )
        with gr.Column(scale=6):
            go_code = gr.Code(
                label="Go (generated)",
                value="",
                language="python",  # "go" not supported by Gradio, python highlighting is close enough
                lines=26,
            )

    with gr.Row():
        python_run = gr.Button("▶ Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value=models[0], show_label=False)
        convert = gr.Button("⚡ Port to Go", elem_classes=["convert-btn"])
        go_run = gr.Button("▶ Run Go", elem_classes=["run-btn", "cpp"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            go_out = gr.TextArea(label="Go result", lines=8, elem_classes=["cpp-out"])

    convert.click(fn=port, inputs=[model, python], outputs=[go_code])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    go_run.click(fn=compile_and_run, inputs=[go_code], outputs=[go_out])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
